![logo](https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/docs/source/_figures/Helmholtz-AI.png?raw=true)

# Transformer Models

In this Notebook we will introduce the main components of Transformer architecture adn combine them into a text classifier. 
We then introduce pretrained models through BERT and DistilBERT. Optional sections cover fin-tuning Distil-BERT for emotion classification and using encoder-decoder Transformers for translation.

---

## Getting Started

### Setup Colab environment

If you installed the packages and requirements on your  machine, you can skip this section and start from the import section.
Otherwise, you can follow and execute the tutorial on your browser. To start working on the notebook, click on the following button. This will open this page in the Colab environment, and you will be able to execute the code on your own.

<a href="https://colab.research.google.com/github/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/xai-for-transformer/1-Tutorial_Transformer_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now that you opened the notebook in Colab, run the next cell. It fetches the tutorials and installs the requirements.

**Note:** Google Colab may ask you to restart the runtime after installing the required packages. If prompted, restart the runtime before continuing with the notebook.


Running the notebook locally, that cell does nothing: you already have the repository and the requirements.

In [1]:
# On Google Colab: fetch the course and install the requirements.
# Running locally this cell does nothing, because you already have the repository
# and installed the requirements yourself.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %cd /content
    !rm -rf XAI-Tutorials
    !git clone --depth 1 --branch main https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials.git
    %cd XAI-Tutorials
    !pip install -r requirements_xai-for-transformer.txt
    %cd xai-for-transformer


### Imports

In [2]:
import math
import evaluate
import numpy as np
import re
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# Use GPU (CUDA), Apple Silicon (MPS), or CPU - whichever is available
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

--------

## Build a Transformer Model

In the following sections, we introduce the main components of the Transformer architecture and implement them step by step in PyTorch. As a running example, we build an encoder-only model for text classification, which assigns one class label to an input sequence. We also introduce the decoder, which we will use in the optional translation section.

***Note: we provide all references [here](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#references).***

### Overview

Transformers largely replaced earlier sequence models (such as recurrent neural networks): their attention mechanism processes all tokens in parallel and captures long-range relationships between them directly, which makes them both faster to train and more accurate. This architecture underpins today's large language models. The video below builds this intuition and visualizes the overall architecture.

*Watch: [Transformer Basic – Overview](https://www.youtube.com/watch?v=aqIZG521uwE&list=PLK2tkofg74mk) · Read more: [Introduction to Transformers](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#introduction-to-transformers)*

### Input Processing

Before an input sequence can be processed by a transformer model, it undergoes several preprocessing steps:

#### Tokenization

Tokenization splits text into smaller units called tokens. Depending on the tokenizer, these may be words, subwords, or characters.

For exmaple a tokenizer converts:

"Natural Language Processing is fascinating."  

into:

["natural", language", "processing", "is", "fascinating", "."]  

We will define a simple tokenizer that lowercases text and separates words and punctuation. We will then build a **vocabulary** from the training texts, assigning each unique token a fixed integer ID. Every occurrence of the same token receives the same ID. The embedding layer introduced next will use these IDs to retrieve vector representations.

Our vocabulary also includes three special tokens:
- [PAD] (padding): fills unused positions with special padding tokens so sequences of different lengths can be combined into a batch of equal-length sequences. Later we will define an **attention mask** to identify these positions so that the attention mechanism can ignores them.
- [UNK] (unknown): represents tokens that are not in the vocabulary. Our tokenizer maps every unseen token to this ID.
- [CLS] (classification): for our classification example, a [CLS] token is added at the beginning of each sequence. Our classifier will use its representation after the encoder layers to predict a label for the whole sequence.

We build the vocabulary once from the training data and keep it fixed when encoding validation, test, or new texts.

This provides a basic word-level tokenizer. More advanced tokenizers can split words into smaller units called subwords. Pretrained models come with an associated tokenizer and vocabulary, as we will see later.

In [3]:
def tokenize(text):
    """Lowercase text and split it into words and punctuation."""
    return re.findall(r"\w+|[^\w\s]", text.lower())


def build_vocab(texts):
    """Build a vocabulary from training texts."""
    vocab = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2}
    unique_tokens = sorted({
        token for text in texts for token in tokenize(text)
    })
    for token in unique_tokens:
        vocab[token] = len(vocab)
    return vocab


def encode(text, vocab):
    """Convert text into token IDs, prepending [CLS]."""
    tokens = ["[CLS]"] + tokenize(text)
    return [vocab.get(token, vocab["[UNK]"]) for token in tokens]

In [4]:
# A tiny training corpus for this demonstration.
training_texts = [
    "Natural Language Processing is fascinating.",
    "Transformers process sequences of tokens.",
]
vocab = build_vocab(training_texts)

text = training_texts[0]
input_ids = torch.tensor([encode(text, vocab)], dtype=torch.long)

print("Tokens:", ["[CLS]"] + tokenize(text))
print("Token IDs:", input_ids)

Tokens: ['[CLS]', 'natural', 'language', 'processing', 'is', 'fascinating', '.']
Token IDs: tensor([[ 2,  7,  6, 10,  5,  4,  3]])



#### Word Embedding

After tokenization, each token ID is mapped to a high-dimensional vector representation called **embedding**. These embeddings serve as numerical input to the transformer and capture useful semantic relationships during training.

An embedding layer acts as a trainable lookup table: each token ID selects one vector, so occurrences of the same token initially receive the same representation. The vectors are randomly initialized and updated during training.

Below, we use PyTorch’s `nn.Embedding` with a small embedding dimension to illustrate this operation.

*Watch: [Transformer Basic – Word Embedding](https://www.youtube.com/watch?v=Z7K2uodMnts&list=PLK2tkofg74mk) · Read more: [Word embedding](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#word-embedding)*


In [5]:
# Create one eight-dimensional vector per vocabulary entry.
embedding = nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=8,
    padding_idx=vocab["[PAD]"],
)

embedded_tokens = embedding(input_ids)

print("Token ID shape:", input_ids.shape)  
print("Embedding shape:", embedded_tokens.shape)
print("First token's embedding:", embedded_tokens[0, 0])

Token ID shape: torch.Size([1, 7])
Embedding shape: torch.Size([1, 7, 8])
First token's embedding: tensor([-1.0003,  0.3936,  0.4751,  0.6605,  0.0849,  1.5989, -0.4116, -0.2384],
       grad_fn=<SelectBackward0>)


The output has shape (`batch_size`, `sequence_length`, `embedding_dimension`). In our example, this is (1, 7, 8): one input sequence, seven tokens (including [CLS] and punctuation), and eight values per token. The embedding layer preserves the batch and sequence dimensions and replaces each token ID with an eight-dimensional vector..

These vectors are initially random and learn useful relationships during training. While token IDs remain fixed, their embedding vectors are updated.
At this stage, each occurrence of the same token retrieves the same vector. Positional encoding and the encoder layers will then incorporate information about its position and surrounding tokens.


#### Positional Encoding

Unlike recurrent or convolutional networks, transformers do not inherently capture the order of tokens in a sequence. To provide positional information, positional encodings are added to the input embeddings. These encodings have the same dimensionality as the embeddings and are typically generated using sine and cosine functions of varying frequencies. This approach assigns a unique representation to each position, enabling the model to distinguish between tokens based on their order within the sequence.

*Watch: [Transformer Basic – Positional Encoding](https://www.youtube.com/watch?v=1s_XUBB1RsY&list=PLK2tkofg74mk) · Read more: [Positional encoding](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#positional-encoding)*

In [6]:
# Define the positional encoding layer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    

### Attention

The attention mechanism is the key component of Transformer models: it lets the model relate different positions in a sequence, so that each token's representation can be informed by the other tokens most relevant to it. Before diving into the implementation, it helps to build an intuition for how attention works.

#### Self-Attention

Within a single attention head, **Scaled Dot-Product Attention** is computed: for every token, the model measures how much it should attend to each other token in the same sequence, and uses those weights to form a context-aware representation of the token.

*Watch: [Transformer Basic – Self Attention](https://www.youtube.com/watch?v=6njflIo3BHI&list=PLK2tkofg74mk) · Read more: [Self-attention](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#self-attention)*

#### Multi-Head Attention

Instead of relying on a single attention operation, Multi-Head Attention uses multiple attention heads, each of which can focus on different aspects of the input and learn distinct patterns or dependencies. The outputs of all attention heads are then concatenated and passed through a linear transformation to produce the final output representation. After reviewing this concept, we will examine the code implementation of Multi-Head Attention in more detail.

*Watch: [Transformer Basic – Multi-Head Attention](https://www.youtube.com/watch?v=99100CbBsck&list=PLK2tkofg74mk) · Read more: [Multi head attention](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#multi-head-attention)*

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

### Putting it all together

The original Transformer has two components, an **encoder** and a **decoder**. Modern models often use only one, depending on the task. Since this course focuses on text *classification*, we will use the **encoder** only.

*Watch: [Transformer Basic – Putting it all together](https://www.youtube.com/watch?v=SE6CH0plnW0&list=PLK2tkofg74mk) · Read more: [Putting them all together](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#putting-them-all-together)*

#### Encoder

The encoder turns the input sequence into a contextualized representation that captures the relationships between tokens. It is a stack of identical layers — the original Transformer uses six — each containing:

* **Multi-Head Self-Attention**, which lets each token attend to all other tokens in the sequence.
* **Feed-Forward Network**, which refines each token's representation independently, applying two linear transformations with a ReLU activation in between.
* **Residual Connections and Layer Normalization**, which stabilize training and help information flow.

The final layer outputs a sequence of vectors that encodes the meaning and context of the input, ready for a downstream task such as classification.

In [8]:
# The position-wise fully connected feed-forward network
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Define the encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

#### Decoder

During generation, the decoder produces an output sequence one token at a time. Each decoder layer uses three main operations:
- **Masked self-attention:** Each target token attends to itself and preceding target tokens. A causal mask prevents attention to future positions.
- **Cross-attention:** The decoder attends to the encoder’s output. Queries come from the decoder, while keys and values come from the encoder.
- **Feed-forward network:** Each token’s representation is processed independently.

*Watch: [Transformer Basic – Masked and Cross Attention](https://www.youtube.com/watch?v=NONly4i_-3Q&list=PLK2tkofg74mk) · Read more: [Masked self-attention and cross-attention](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#masked-self-attention-and-cross-attention)*

In [9]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, source_mask, target_mask):
        # Masked self-attention over the target sequence.
        attn_output = self.self_attn(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Cross-attention over the encoder's output:
        # queries come from the decoder, keys and values from the encoder.
        # The source mask excludes padding in the input sequence.
        cross_attn_output = self.cross_attn(
            x, encoder_output, encoder_output, source_mask
        )
        x = self.norm2(x + self.dropout(cross_attn_output))

        # 
        # Position-wise feed-forward network.
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

Here, `x` contains the target sequence’s representations, initially obtained through token embedding and positional encoding. `encoder_output` contains the contextual representations of the source sequence.

Two masks are supplied to the layer:
- `source_mask`: prevents cross-attention to padding positions in the source sequence.
- `target_mask`: combines target padding information with a **causal mask**, preventing self-attention to future target positions.
This class implements one decoder layer. A complete encoder–decoder Transformer stacks multiple such layers and adds an output projection to predict target tokens.

>Note: Our classification task uses only the encoder, so the decoder is not included in the classifier assembled below. In the **Extra Material: Transformers for Translation** section at the end of this notebook, we combine these components into a complete encoder-decoder Transformer and demonstrate translation fine-tuning usinf a separate pretrained model.

### Assemble the final Transformer Classifier

We now combine the componentes introduced above into a complete encoder-only Transformer classifier.

The model receives the token IDs prepared by our tokenizer. Tokenization and padding happen outside the model; the following operations happen inside it:

- **Attention mask generation:** We first create the padding mask introduced in the tokenization section. Since our vocabulary assigns [PAD] the ID 0, `input_ids!=0` identifies the non-padding positions. This mask is passed to each encoder layer to prevent the attention mechanism from focusing on padding tokens, ensuring that only meaningful tokens contribute to the learned representations.

- **Token Embedding:** The input tokens are mapped to dense vector representations through an embedding layer (`embedding`). 

- **Positional encoding:** A `PositionalEncoding` layer adds positional information to the embedding, enabling the model to distinguish between different positions in the sequence.

- **Encoder layers:** The model contains a stack of encoder layers, defined by `num_layers`. Each layer applies multi-head self-attention and a feed-forward network, with residual connections and layer normalization. These layers progressively transform the input representations into contextual representations that capture relationships between non-padding tokens.

- **Classification head:** Our input preparation prepends [CLS] to each sequence. After the encoder stack, its final contextual representation is selected as the sequence-level representation and passed through a fully connected layer to produce one score (logit) per class.

> **Note:** Our simplified implementation uses the representation of the first token as a sequence-level representation. In BERT and DistilBERT, a special classification token (`[CLS]`) is prepended to every input sequence, and its final contextual representation is used by the classification head.

In [10]:
# Define the transformer classifier model
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers,
                 d_ff, max_seq_length, num_classes, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, input_ids):
        # True for real tokens, False for padding tokens (ID 0).
        mask = input_ids != 0

        # Add dimensions so the same padding mask can be used
        # by every attention head and for every token in the sequence
        return mask.unsqueeze(1).unsqueeze(2)

    def forward(self, input_ids):
        mask = self.generate_mask(input_ids)

        x = self.embedding(input_ids)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        for encoder_layer in self.encoder_layers:
            x = encoder_layer(x, mask)

        cls_representation = x[:, 0, :]
        logits = self.classifier(cls_representation)

        return logits

### Parametrize the Transformer Classifier

To instantiate the Transformer classifier, we define the main architectural parameters that control the size and complexity of the model:

- `vocab_size`: The size of the vocabulary, i.e., the total number of unique tokens (words or subwords) that the model can process.
- `d_model`: The dimensionality of the token embeddings and hidden representations throughout the model.
- `num_heads`: The number of attention heads used in each multi-head self-attention layer.
- `num_layers`: The number of stacked encoder layers.
- `d_ff`: The dimensionality of the hidden layer in the feed-forward network contained in each encoder layer.
- `max_seq_length`: The maximum input sequence length supported by the positional encoding.
- `num_classes`: The number of output classes for the classification task.
- `dropout`: The dropout probability used as a regularization technique to reduce overfitting.

The following parameters define a small Transformer classifier for demonstration purposes.

In [11]:
vocab_size = len(vocab)
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
num_classes = 2 # Binary classification for this custom-model example.
dropout = 0.1

In [12]:
transformer_classifier = TransformerClassifier(
    vocab_size,
    d_model,
    num_heads,
    num_layers,
    d_ff,
    max_seq_length,
    num_classes,
    dropout
).to(device)

### Run the classifier
We can now pass our tokenized input ("*Natural Language Processing is fascinating*") through the assembled model. For each input sequence, the classifier returns two scores, called logits (one per class). Applying softmax converts these scores into probabilities, and selecting the highest score gives the predicted class. 

For a binary sentiment task, we could define class 0 as “positive” and class 1 as “negative.” However, the model has not been trained on labeled examples, so it has not learned these meanings and its predictions are not yet meaningful. This example demonstrates how to use the complete classifier. Changing the sentence may still produce the same predicted class; words outside our vocabulary are mapped to [UNK].

In [13]:
# Reuse our original sentence, or change it to try another input.
text = "Natural Language Processing is fascinating."

# Encode the current text using the existing vocabulary.
input_ids = torch.tensor(
    [encode(text, vocab)],
    dtype=torch.long,
)

# Switch to evaluation mode to disable dropout during inference.
transformer_classifier.eval()

with torch.no_grad():
    logits = transformer_classifier(input_ids.to(device))
    probabilities = torch.softmax(logits, dim=-1)
    predicted_classes = probabilities.argmax(dim=-1)


print("Input text:", text)
print("  Class probabilities:", probabilities[0].cpu().tolist())
print("  Predicted class:", predicted_classes[0].item())

Input text: Natural Language Processing is fascinating.
  Class probabilities: [0.4562543034553528, 0.5437456965446472]
  Predicted class: 1


## BERT

The previous sections introduced the original Transformer architecture and demonstrated how its main components can be assembled into an encoder-only Transformer classifier.

In practice, however, modern NLP applications rarely train Transformer models from scratch. Instead, they rely on **pretrained language models**, which are first trained on large text corpora and later adapted to specific downstream tasks through fine-tuning.

One of the most influential encoder-only Transformer models is **BERT (Bidirectional Encoder Representations from Transformers)**. Unlike the original encoder-decoder Transformer, BERT consists only of the encoder stack and is designed to learn contextual representations of text by attending to both the left and right context of every token simultaneously.

*Watch: [Transformer Basic – BERT](https://www.youtube.com/watch?v=s0OQoRuNPTg&list=PLK2tkofg74mk) · Read more: [Bidirectional Encoder Representations from Transformers (BERT)](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#bidirectional-encoder-representations-from-transformers-bert)*

In this course, we will use **DistilBERT**, a lightweight version of BERT that retains most of its performance while requiring fewer parameters and less computational resources. This makes it particularly suitable for educational purposes and explainability analyses.

### Loading a pretrained DistilBERT model

In [14]:
checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=6
    ).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Extra Material 1:  Fine-tune a DistillBERT for Emotion Classification

In the subsequent sections we will show you how to fine-tune a pre-trained Transformer model from [Hugging Face](https://huggingface.co/docs/transformers/index) on a text-classification task.

Our goal is to fine-tune the pre-trained **DistilBERT** model for an emotion-classification task, i.e. predicting the emotion expressed in a short piece of text.
We use `distilbert-base-uncased`, the lightweight version of BERT introduced above.
You can find the model card [here](https://huggingface.co/distilbert/distilbert-base-uncased).

In [15]:
checkpoint = "distilbert-base-uncased"

The first step for fine-tuning a model is to load a dataset and prepare it for the classification task.
Here we use the [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) dataset, which contains English tweets labeled with one of six emotions: *sadness, joy, love, anger, fear,* and *surprise*.

In [16]:
emotion = load_dataset("dair-ai/emotion", "split")

labels = emotion["train"].features["label"].names
num_labels = len(labels)
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

labels

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

Next, we initialize a tokenizer based on the specified model checkpoint, which converts text into token IDs that the model can process.
We then define a pre-processing function that tokenizes each example's text (truncating long sequences) and apply it to the whole dataset.

In [17]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_emotion = emotion.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

For setting up the training we define the training arguments using the `TrainingArguments` class from the Hugging Face Transformers library.
The training arguments include settings such as the learning rate, batch size, number of epochs, evaluation strategy, and weight decay.

In [18]:
training_args = TrainingArguments(
    output_dir="distilbert-emotion",
    eval_strategy="epoch", #eval_strategy in the new version
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    num_train_epochs=3,
    save_total_limit=2,
    push_to_hub=False,
)

In addition to the training arguments, we define the evaluation metrics for our model.
For a classification task we report **accuracy** and the **weighted F1-score**, which balances precision and recall across the (imbalanced) emotion classes.

In [19]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }

Now we are ready to fine-tune DistilBERT for our emotion-classification task.
We initialize the model from the pre-trained checkpoint, adding a classification head sized to the six emotion labels.
The data collator dynamically pads the input sequences to the same length within each batch.

After initializing the `Trainer`, we can start the training/fine-tuning of the model.

*Note: for demonstration purposes we select only a small subsample of the train and validation data. The fully fine-tuned weights are provided below.*

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_emotion["train"].select(range(200)),    # small subsample for demonstration
    eval_dataset=tokenized_emotion["validation"].select(range(50)),  # small subsample for demonstration
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.603174,0.260000,0.107302
2,No log,1.511500,0.260000,0.107302
3,No log,1.492360,0.260000,0.107302


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=39, training_loss=1.623790251903045, metrics={'train_runtime': 32.8269, 'train_samples_per_second': 18.278, 'train_steps_per_second': 1.188, 'total_flos': 7366954726368.0, 'train_loss': 1.623790251903045, 'epoch': 3.0})

### Load the fully fine-tuned model

The demonstration above only fine-tunes on a small subsample, so its predictions are not yet reliable.
Fully fine-tuning DistilBERT on the complete emotion dataset takes ~20 minutes, so we provide the resulting weights as a downloadable asset.
The cell below loads them - reusing a local copy in `../data/` if one is already present, otherwise downloading it - and the rest of the tutorials use this fine-tuned model.

*The training script and metrics for these weights are in [`distilbert-emotion-training/`](./distilbert-emotion-training).*

In [21]:
import os
import zipfile
import requests

# Fine-tuned DistilBERT emotion weights, hosted as a GitHub Release asset.
# NOTE: the download link becomes active once the weights Release is published.
url = "https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/releases/download/distilbert-emotion-weights/distilbert_emotion_weights.zip"

weights_path = "../data/distilbert-emotion"
zip_path = "../data/distilbert_emotion_weights.zip"
os.makedirs("../data", exist_ok=True)

if not os.path.exists(weights_path):
    # Reuse a local zip if it is already there (e.g. shared for testing),
    # otherwise download it from the Release.
    if not os.path.exists(zip_path):
        with open(zip_path, "wb") as f:
            f.write(requests.get(url).content)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("../data/")

model = AutoModelForSequenceClassification.from_pretrained(weights_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(weights_path)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Extra Material 2: Transformers for Translation

Our classifier produces one label per input sequence. Translation instead requires generating an output sequence, potentially of a different length from the input.

We first combine the encoder and decoder components introduced earlier into a complete encoder–decoder Transformer. We then demonstrate fine-tuning a pretrained model for English-to-German translation.


### Assemble an encoder–decoder Transformer
The model receives two sequences:

- Source sequence: the input text to translate.
- Target sequence: the preceding output tokens available to the decoder.

The source and target sequences have separate embedding layers. After positional encoding, the source representations pass through the encoder.
The decoder processes the target representations using masked self-attention and cross-attention over the encoder's output.

Two masks control attention:

- Source mask: excludes source padding positions in encoder self-attention and decoder cross-attention.
- Target mask: excludes target padding positions and future positions in decoder self-attention.

Finally, a linear layer maps each decoder representation to one score per token in the target vocabulary.

In [22]:
class TransformerTranslator(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, d_model, num_heads, num_layers,
                 d_ff, max_seq_length, dropout, source_pad_id=0, target_pad_id=0,):
        super().__init__()

        self.source_pad_id = source_pad_id
        self.target_pad_id = target_pad_id

        self.encoder_embedding = nn.Embedding(source_vocab_size, d_model, padding_idx=source_pad_id)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model, padding_idx=target_pad_id)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.output_layer = nn.Linear(d_model, target_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_masks(self, source_ids, target_ids):
        # Identify non-padding source positions.
        source_mask = ((source_ids != self.source_pad_id).unsqueeze(1).unsqueeze(2))

        # Identify non-padding target positions.
        target_padding_mask = ((target_ids != self.target_pad_id).unsqueeze(1).unsqueeze(2))

        # Allow attention only to the current and earlier positions.
        target_length = target_ids.size(1)
        causal_mask = torch.ones(target_length, target_length, dtype=torch.bool, device=target_ids.device,).tril()

        target_mask = target_padding_mask & causal_mask
        return source_mask, target_mask

    def forward(self, source_ids, target_ids):
        source_mask, target_mask = self.generate_masks(source_ids, target_ids)

        # Embed the source sequence and add positional information.
        encoder_output = self.dropout(self.positional_encoding(self.encoder_embedding(source_ids)))

        for encoder_layer in self.encoder_layers:
            encoder_output = encoder_layer(encoder_output, source_mask)

        # Embed the target sequence and add positional information.
        decoder_output = self.dropout(self.positional_encoding(self.decoder_embedding(target_ids)))

        for decoder_layer in self.decoder_layers:
            decoder_output = decoder_layer(decoder_output, encoder_output, source_mask, target_mask,)

        return self.output_layer(decoder_output)

For sequence generation, we introduce two additional special tokens:
- [BOS] (beginning of sequence): marks the start of the target sequence and provides the first input to the decoder.
- [EOS] (end of sequence): marks the end of the target sequence. During generation, predicting this token indicates that the output is complete.

For example, the target sentence "Hallo Welt." becomes:

[BOS] hallo welt . [EOS]

During training, we shift this sequence to form the decoder inputs and expected outputs:

- Decoder input: [BOS] hallo welt .
- Expected output: hallo welt . [EOS]

Each decoder position therefore learns to predict the next token.

In [23]:
source_text = "Hello world."
target_text = "Hallo Welt."

# Reuse our basic tokenizer, but build translation-specific vocabularies.
def build_translation_vocab(texts):
    vocabulary = {"[PAD]": 0, "[UNK]": 1, "[BOS]": 2, "[EOS]": 3}
    for token in sorted({
        token for text in texts for token in tokenize(text)
    }):
        vocabulary[token] = len(vocabulary)
    return vocabulary


source_vocab = build_translation_vocab([source_text])
target_vocab = build_translation_vocab([target_text])

# Prepare source IDs without the classification-specific encode function.
source_ids = torch.tensor([[
    source_vocab[token] for token in tokenize(source_text)
]], dtype=torch.long)

# Create the full target sequence, then shift it.
target_sequence = (
    [target_vocab["[BOS]"]]
    + [target_vocab[token] for token in tokenize(target_text)]
    + [target_vocab["[EOS]"]]
)
target_ids = torch.tensor([target_sequence], dtype=torch.long)

decoder_input_ids = target_ids[:, :-1]
expected_output_ids = target_ids[:, 1:]

translation_transformer = TransformerTranslator(
    source_vocab_size=len(source_vocab),
    target_vocab_size=len(target_vocab),
    d_model=64,
    num_heads=4,
    num_layers=2,
    d_ff=256,
    max_seq_length=100,
    dropout=0.1,
).to(device)

translation_transformer.eval()

with torch.no_grad():
    translation_logits = translation_transformer(
        source_ids.to(device),
        decoder_input_ids.to(device),
    )

print("Source:", source_text)
print("Target:", target_text)
print("Decoder input shape:", decoder_input_ids.shape)
print("Expected output shape:", expected_output_ids.shape)
print("Output shape:", translation_logits.shape)
# (1, 4, len(target_vocab))

Source: Hello world.
Target: Hallo Welt.
Decoder input shape: torch.Size([1, 4])
Expected output shape: torch.Size([1, 4])
Output shape: torch.Size([1, 4, 7])


>Note: The special tokens [BOS] and [EOS] belong to our custom translation example. Pretrained models use their own conventions. For example the T5 that we will see in the next section uses its padding token to start decoding rather than a separate [BOS] token.

### Fine-tune a pretrained model for translation

The previous example demonstrates how our encoder and decoder components fit together. Training this model from scratch would require a parallel corpus and a training procedure.

Instead, we now load a separate pretrained encoder–decoder model, T5-small, and fine-tune it for English-to-German translation using the `opus_books` dataset. This dataset contains parallel English and German texts. We set English as the source language and German as the target language.

T5 uses a text prefix to identify the task. We prepend "translate English to German: " to each English input. The corresponding German text is the target output.

> Note: T5 includes its own tokenizer and vocabulary. The custom vocabularies and randomly initialized model above are not used in this fine-tuning example.

In [24]:
translation_checkpoint = "google-t5/t5-small"

books = load_dataset("opus_books", "de-en")
books = books["train"].train_test_split(test_size=0.2, seed=42)

source_language = "en"
target_language = "de"
prefix = "Translate English to German: "

README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Using the latest cached version of the dataset since opus_books couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'de-en' at /Users/donatella.cea/.cache/huggingface/datasets/opus_books/de-en/0.0.0/1f9f6191d0e91a3c539c2595e2fe48fc1420de9b (last modified on Mon Nov  9 12:26:36 2026).


Next, we initialize a tokenizer based on the specified model checkpoint, which will convert text into token IDs that the model can process  
and define a pre-processing function that prepares batches of translation examples for the translation task by:
- Prepending the task-specific prefix to the source language sentences.
- Extracting the corresponding target language sentences.
- Tokenizing both the inputs and targets into sequences of tokens suitable for input to the t5-small model.

In [25]:
translation_tokenizer = AutoTokenizer.from_pretrained(translation_checkpoint)

def preprocess_translation(examples):
    inputs = [prefix + example[source_language] for example in examples["translation"]]
    targets = [example[target_language] for example in examples["translation"]]
    model_inputs = translation_tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

tokenized_books = books.map(preprocess_translation, batched=True)

Map:   0%|          | 0/41173 [00:00<?, ? examples/s]

Map:   0%|          | 0/10294 [00:00<?, ? examples/s]

For setting up the model we need to define the training arguments for our t5-small model using the `Seq2SeqTrainingArguments` class from the Hugging Face Transformers library.  
The training arguments include settings for the learning rate, batch size, number of epochs, evaluation strategy, model checkpointing, etc.

In [26]:
translation_training_args = Seq2SeqTrainingArguments(
    output_dir="t5-english-to-german",
    eval_strategy="epoch",
    learning_rate=2e-3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=False, 
    push_to_hub=False,
)

In addition to the tarining arguments, we need to define the evaluation metrics for our model.  
Here, we use the SacreBLEU metric, which is a standard evaluation metric for machine translation tasks,  
providing a BLEU (Bilingual Evaluation Understudy) score, which measures the quality of text generated by the model compared to reference translations.

In [27]:
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_translation_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = translation_tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, translation_tokenizer.pad_token_id)
    decoded_labels = translation_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != translation_tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens) # average number of non-pad tokens in predictions (model output)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

Using the latest cached version of the module from /Users/donatella.cea/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--sacrebleu/28676bf65b4f88b276df566e48e603732d0b4afd237603ebdf92acaacf5be99b (last modified on Mon Nov  9 12:26:39 2026) since it couldn't be found locally at evaluate-metric--sacrebleu, or remotely on the Hugging Face Hub.


We initialize T5-small from its pretrained checkpoint and configure a `Seq2SeqTrainer` for translation fine-tuning. The data collator dynamically pads input and target sequences within each batch.

This example illustrates the same fine-tuning principle used earlier with DistilBERT, but for sequence generation rather than classification.

>Note: Running training is optional and is not required for any subsequent part of the tutorial. We leave the training call commented out because it can take substantial time, even with the small dataset subsets used here.

In [28]:
translation_model = AutoModelForSeq2SeqLM.from_pretrained(translation_checkpoint).to(device)
translation_collator = DataCollatorForSeq2Seq(tokenizer=translation_tokenizer, model=translation_checkpoint)

translation_trainer = Seq2SeqTrainer(
    model=translation_model,
    args=translation_training_args,
    train_dataset=tokenized_books["train"].select(range(200)), # we only select a small subsample for demonstration purposes
    eval_dataset=tokenized_books["test"].select(range(50)), # we only select a small subsample for demonstration purposes
    processing_class=translation_tokenizer, 
    data_collator=translation_collator,
    compute_metrics=compute_translation_metrics,
)
# Uncomment this line to train the translation model. Note that training can take a long time depending on the hardware.
# translation_trainer.train()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]